In [24]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFMCS

import joblib

ROOT = Path("../..").resolve()  
SIGNOFF = ROOT / "hergbench_stage1_signoff"  


REPORT_JSON = SIGNOFF / "inputs" / "reports" / "runs" / "lead_reports" / "20ae67ffe6fb365715533270623337b04f656686" / "report.json"
MODEL_PATH  = SIGNOFF / "models_calibrators" / "model_cluster_seed11.joblib"
META_PATH   = SIGNOFF / "models_calibrators" / "model_meta_cluster_seed11.json"

# optional train-proximity context totally opt.
# DATASET_PATH = ROOT / "data" / "processed" / "herg_clean.csv"


In [13]:
rep = json.loads(Path(REPORT_JSON).read_text(encoding="utf-8"))

base_std = rep.get("base_smiles_std") or rep.get("base_smiles_raw")
threshold = rep.get("threshold", None)
max_sim_to_train = rep.get("max_sim_to_train", None)

cfs = rep.get("counterfactuals", [])
print("Base:", base_std)
print("n counterfactuals:", len(cfs))
print("threshold:", threshold, "max_sim_to_train:", max_sim_to_train)


Base: O=C1NCCN1CC[NH+]1CC=C(c2cn(-c3ccc(F)cc3)c3ccc(Cl)cc23)CC1
n counterfactuals: 3
threshold: 0.51 max_sim_to_train: 0.7


In [ ]:
# Parse molecules + define fingerprints

def mol_from_smiles(smi: str):
    m = Chem.MolFromSmiles(smi)
    if m is None:
        raise ValueError(f"RDKit failed to parse SMILES: {smi}")
    return m

base_mol = mol_from_smiles(base_std)

def morgan_fp(mol, radius=2, nBits=2048):
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits)

base_fp = morgan_fp(base_mol)


Compute similarity-to-base for each counterfactual. This step is crucial in this test as we already know similarity to train but we also need similarity to the base molecule.


**Question/hypothesis test # 1. If the generated hypotheses are indeed actionable can the similarity to the base be confirmed? (reports generated only inform similarity to train)**


In [15]:
rows = []
for i, cf in enumerate(cfs, start=1):
    smi = cf.get("smiles_std") or cf.get("raw_smiles")
    m = Chem.MolFromSmiles(smi)
    if m is None:
        continue
    sim_to_base = DataStructs.TanimotoSimilarity(base_fp, morgan_fp(m))
    rows.append({
        "idx": i,
        "smiles": smi,
        "tier_num_std": cf.get("tier_num_std"),
        "tier_label_std": cf.get("tier_label_std"),
        "tier_raw": cf.get("tier_raw"),
        "p_std": cf.get("p_std"),
        "delta_p_std": cf.get("delta_p_std"),
        "similarity_to_train": cf.get("similarity_to_train"),
        "similarity_to_base": sim_to_base,
        "sascore": cf.get("sascore"),
        "logp": cf.get("logp"),
        "qed": cf.get("qed"),
        "alert": cf.get("alert"),
    })

df = pd.DataFrame(rows).sort_values(["tier_num_std","p_std","similarity_to_base"], ascending=[True, True, False])
df.head(10)


,idx,smiles,tier_num_std,tier_label_std,tier_raw,p_std,delta_p_std,similarity_to_train,similarity_to_base,sascore,logp,qed,alert
0,1,CC1=CC=Cn2cc(C3=CCN(CCN4CCNC4=O)CC3)c3cc(Cl)cc...,1,flip,flip,0.406071,0.255189,0.500000,0.336957,3.498544,4.3821,0.793708,False
1,2,O=C1NCCN1CCN1CC=C(c2cn3c4c(cc(Cl)cc24)C(CF)C=C...,1,flip,flip,0.411597,0.249663,0.525641,0.355556,4.019369,3.9464,0.804431,False
2,3,O=C1NCCN1CCN1CC=C(c2cn3c4c(cc(Cl)cc24)C=CC(F)=...,1,flip,flip,0.422148,0.239112,0.567568,0.383721,3.490131,4.2892,0.791878,False


In [19]:
df[["smiles", "similarity_to_base", "similarity_to_train"]]

,smiles,similarity_to_base,similarity_to_train
0,CC1=CC=Cn2cc(C3=CCN(CCN4CCNC4=O)CC3)c3cc(Cl)cc...,0.336957,0.500000
1,O=C1NCCN1CCN1CC=C(c2cn3c4c(cc(Cl)cc24)C(CF)C=C...,0.355556,0.525641
2,O=C1NCCN1CCN1CC=C(c2cn3c4c(cc(Cl)cc24)C=CC(F)=...,0.383721,0.567568


**Key nuance here is**

- `similarity_to_train` tells us “how in-distribution” the CF is (AD-ish).

- `similarity_to_base` tells us “how plausible as a local lead-opt edit” it is

In [ ]:
assert df["similarity_to_base"].notna().all()
print(df["similarity_to_base"].describe())

# basic must passes.


count    3.000000
mean     0.358744
std      0.023545
min      0.336957
25%      0.346256
50%      0.355556
75%      0.369638
max      0.383721
Name: similarity_to_base, dtype: float64


_good to go_

**Question/hypothesis test # 2. Are these consistently low-risk across nearby thresholds or across a second seed/model? (Stage 2 will help.)**

2.1 Threshold sweep around the chosen cutoff (quick, strong)

**This answers: Does the CF remain low-risk across plausible threshold choices?**

In [20]:
p_base = rep.get("p_base_std") or rep.get("p_base_raw")
df["p_base"] = p_base

# Sweep thresholds around the op point
thr_grid = np.round(np.linspace(0.40, 0.70, 31), 3)

def classify(p, thr):
    return int(p >= thr)

# For each CF this snippet gives us fraction of thresholds where it stays below threshold (i.e., predicted non-toxic)
stability = []
for _, r in df.iterrows():
    p = float(r["p_std"])
    below = [(p < thr) for thr in thr_grid]
    stability.append(np.mean(below))

df["below_thr_fraction_0p40_0p70"] = stability

# margin at the operating threshold (if present at all)
if threshold is not None:
    df["margin_at_report_thr"] = float(threshold) - df["p_std"]

df[["idx","tier_num_std","p_std","delta_p_std","similarity_to_base","below_thr_fraction_0p40_0p70","margin_at_report_thr"]].head(10)


,idx,tier_num_std,p_std,delta_p_std,similarity_to_base,below_thr_fraction_0p40_0p70,margin_at_report_thr
0,1,1,0.406071,0.255189,0.336957,0.967742,0.103929
1,2,1,0.411597,0.249663,0.355556,0.935484,0.098403
2,3,1,0.422148,0.239112,0.383721,0.903226,0.087852


**Inferences**

- below_thr_fraction_0p40_0p70 is not tiny (>0.6 is more than likely meaningful stability) .,which we have. 
- margin_at_report_thr comfortably positive (>0.05 is a decent heuristic).

_good to go_

2.2 Cross-model robustness (stronger, but requires a second model)

**This answers: Does a second baseline model trained on the same split agree with the flip?**

_Note: unfortunately, at our stage the second D_MPNN model is not ready, so a reasonable heuristic at this stage is to train a second  baseline model (xgboost), invoked via cli using a different seed_

In [22]:
import os, subprocess

cmd = ["python", "-m", "hergbench.cli", "stage1",
       "--config", "configs/stage1_reference_cluster_seed11_seed22.yaml",
       "--skip-counterfactuals"]

env = dict(os.environ, PYTHONPATH="./src")
subprocess.run(cmd, check=True, env=env)


[2026-01-19 04:00:56,955] INFO - Starting Stage 1 run: 2026-01-19_040056_seed22_seed22_split11_qc
[2026-01-19 04:00:56,956] INFO - Global seed set to 22
[2026-01-19 04:00:57,333] INFO - Wrote run metadata and resolved config


Downloading...
100%|██████████| 50.2k/50.2k [00:00<00:00, 207kiB/s] 
Loading...
Done!


[2026-01-19 04:01:02,672] INFO - Fetched TDC dataset 'hERG' with 655 rows and columns: ['Drug_ID', 'Drug', 'Y']
[2026-01-19 04:01:02,675] INFO - Saved raw dataset to data/raw/tdc_herg_raw.csv


[04:01:03] Can't kekulize mol.  Unkekulized atoms: 3 7
[04:01:04] Tautomer enumeration stopped at 626 tautomers: max transforms reached
[04:01:05] Tautomer enumeration stopped at 318 tautomers: max transforms reached
[04:01:06] WARNING: not removing hydrogen atom without neighbors
[04:01:06] WARNING: not removing hydrogen atom without neighbors
[04:01:06] WARNING: not removing hydrogen atom without neighbors
[04:01:06] WARNING: not removing hydrogen atom without neighbors
[04:01:06] WARNING: not removing hydrogen atom without neighbors
[04:01:06] WARNING: not removing hydrogen atom without neighbors
[04:01:07] Can't kekulize mol.  Unkekulized atoms: 4 9


[2026-01-19 04:01:07,200] INFO - Standardized molecules: kept=655, dropped_invalid=0
[2026-01-19 04:01:07,203] WARNING - Dropping 3 SMILES with conflicting labels after standardization.
[2026-01-19 04:01:07,204] INFO - Deduplicated: 649 -> 635 unique molecules
[2026-01-19 04:01:07,206] INFO - Saved processed dataset to data/processed/herg_clean.csv
[2026-01-19 04:01:07,331] INFO - Stage1: split=cluster seed=11
[2026-01-19 04:01:07,331] INFO - Using sigmoid calibration for cluster split to reduce overfitting risk.
[2026-01-19 04:01:07,480] INFO - scale_pos_weight=0.399 (train pos=363 neg=145)


[I 2026-01-19 04:01:07,483] A new study created in memory with name: no-name-053ed909-8dfe-4064-aada-8f90805531a2
[I 2026-01-19 04:01:07,807] Trial 0 finished with value: 0.8709002804885699 and parameters: {'learning_rate': 0.020319896452155897, 'max_depth': 6, 'min_child_weight': 1.7623923645636188, 'subsample': 0.9436727994085377, 'colsample_bytree': 0.6684646214446693, 'reg_alpha': 1.1214389743303267e-05, 'reg_lambda': 2.7214195459556954e-06, 'gamma': 3.455206752247981}. Best is trial 0 with value: 0.8709002804885699.
[I 2026-01-19 04:01:08,409] Trial 1 finished with value: 0.8842250428769042 and parameters: {'learning_rate': 0.02116236478114709, 'max_depth': 9, 'min_child_weight': 0.5160191078694062, 'subsample': 0.8244814786651999, 'colsample_bytree': 0.9254904749660786, 'reg_alpha': 0.050804556500026264, 'reg_lambda': 5.035031562929253e-07, 'gamma': 0.03070433173049114}. Best is trial 1 with value: 0.8842250428769042.
[I 2026-01-19 04:01:08,547] Trial 2 finished with value: 0.898

[2026-01-19 04:01:22,640] INFO - Best val aucpr=0.9406 with params={'learning_rate': 0.2624440644733788, 'max_depth': 6, 'min_child_weight': 0.7500022101651357, 'subsample': 0.7352182303412644, 'colsample_bytree': 0.7658947973440318, 'reg_alpha': 0.00011525159216281488, 'reg_lambda': 7.027996622799191e-07, 'gamma': 1.2064975020615174}
[2026-01-19 04:01:26,323] INFO - Wrote reports/runs/2026-01-19_040056_seed22_seed22_split11_qc/tables/benchmark_results.csv
[2026-01-19 04:01:26,842] INFO - Wrote reports/runs/2026-01-19_040056_seed22_seed22_split11_qc/tables/applicability_domain_bins.csv
[2026-01-19 04:01:26,843] INFO - Stage 1 completed.
[2026-01-19 04:01:26,843] INFO - Stage 1 completed successfully. Artifacts at: reports/runs/2026-01-19_040056_seed22_seed22_split11_qc


CompletedProcess(args=['python', '-m', 'hergbench.cli', 'stage1', '--config', 'configs/stage1_reference_cluster_seed11_seed22.yaml', '--skip-counterfactuals'], returncode=0)

2.2.1 Load models + recompute p for base/CFs (model-based check)



In [25]:

from hergbench.features.fingerprints import FingerprintConfig, smiles_list_to_fps, fps_to_numpy

bundle = joblib.load(MODEL_PATH)
meta = json.loads(Path(META_PATH).read_text())

# Bundle format used in stage1 pipe.
if isinstance(bundle, dict) and "cal_model" in bundle:
    cal_model = bundle["cal_model"]
    fp_cfg = bundle.get("fp_cfg", {"radius": 2, "n_bits": 2048})
else:
    cal_model = bundle
    fp_cfg = {"radius": 2, "n_bits": 2048}

fp_cfg = FingerprintConfig(radius=int(fp_cfg["radius"]), n_bits=int(fp_cfg["n_bits"]))

def predict_p(smiles_list):
    fps, valid_idx = smiles_list_to_fps(smiles_list, fp_cfg)
    X = fps_to_numpy(fps)
    out = np.full(len(smiles_list), np.nan, dtype=float)
    if len(valid_idx):
        out[valid_idx] = cal_model.predict_proba(X)[:, 1]
    return out

base_p_model = float(predict_p([base_std])[0])
df["p_model"] = predict_p(df["smiles"].tolist())
df["delta_p_model_vs_base"] = base_p_model - df["p_model"]

df[["idx","p_std","p_model","delta_p_std","delta_p_model_vs_base","similarity_to_base","tier_num_std"]].head(10)


,idx,p_std,p_model,delta_p_std,delta_p_model_vs_base,similarity_to_base,tier_num_std
0,1,0.406071,0.406071,0.255189,0.255189,0.336957,1
1,2,0.411597,0.411597,0.249663,0.249663,0.355556,1
2,3,0.422148,0.422148,0.239112,0.239112,0.383721,1


**3 structure-level questions:**

   - Is the core scaffold preserved? (series continuity)

   - Are edits “small” (local transformations rather than hopping series)?

   - Do properties move in obviously-dangerous directions? (basic developability sanity)

In [ ]:
def murcko_smiles(mol):
    scaf = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaf) if scaf is not None else None

base_scaf = murcko_smiles(base_mol)

def mcs_ratio(m1, m2):
    res = rdFMCS.FindMCS([m1, m2], ringMatchesRingOnly=True, completeRingsOnly=True, timeout=5)
    patt = Chem.MolFromSmarts(res.smartsString) if res.smartsString else None
    if patt is None:
        return 0.0
    n_mcs = patt.GetNumAtoms()
    return n_mcs / max(1, m1.GetNumAtoms())

scaf_same = []
mcs_ratios = []
for smi in df["smiles"]:
    m = mol_from_smiles(smi)
    scaf_same.append(murcko_smiles(m) == base_scaf)
    mcs_ratios.append(mcs_ratio(base_mol, m))

df["murcko_scaffold_same"] = scaf_same
df["mcs_ratio_to_base"] = mcs_ratios

df[["idx","similarity_to_base","murcko_scaffold_same","mcs_ratio_to_base","sascore","logp","qed","alert","tier_num_std"]].head(10)
